In [ ]:
import pandas as pd
from pathlib import Path

import datetime
import os

import warnings
warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2
from utils.simulation_utils import filter_data, get_simulator, generate_session_results, TypeScenario

Select your parameters below

In [ ]:
months = range(1, 2)
year = 2023
scenario: TypeScenario = "peak_forecast"
run_verbose = True
final_verbose = True
filter_date_greater_than: pd.Timestamp | None = pd.to_datetime("2023-01-15 00:40:33")
initial_running_peak: float = 42.02

Run the simulation

In [ ]:
sessions_file = Path(__name__).resolve().parents[1] / "data" / "Sessions3.csv"
sessions_df = pd.read_csv(sessions_file)
sessions_df = sessions_df.sort_values(by="startChargeTime")

# Create output folder for this simulation
current_time = datetime.datetime.now()
time_str = current_time.strftime("%Y-%m-%d_%H-%M-%S")
folder_path = f"results/{scenario}/{time_str}"
os.makedirs(folder_path, exist_ok=True)

for month in months:
    # function to filter data in the same way for each scenario
    test_df = filter_data(
        sessions_df, month, year, "all_scheduled", filter_date_greater_than
    )  # .head(30)

    # function to run a scenario (returns a child of BaselineSimulator)
    sim = get_simulator(
        test_df,
        scenario,
        verbose=run_verbose,
        initial_running_peak=initial_running_peak,
    )

    # Run simulation and save results and append to the summary
    results_file_name = f"{folder_path}/{month}_{year}_{scenario}.csv"
    summary_file_name = f"{folder_path}/summary.csv"
    aggregate_power_profile_file_name = f"{folder_path}/aggregate_power_profile.csv"
    session_results = generate_session_results(
        sim,
        month,
        results_file_name,
        summary_file_name,
        aggregate_power_profile_file_name,
        visualize=final_verbose,
        verbose=final_verbose,
    )

Run scenario "smooth_dc_penalty"

In [ ]:
sessions_file = Path(__name__).resolve().parents[1] / "data" / "Sessions3.csv"
sessions_df = pd.read_csv(sessions_file)
sessions_df = sessions_df.sort_values(by="startChargeTime")

# Create output folder for this simulation
current_time = datetime.datetime.now()
time_str = current_time.strftime("%Y-%m-%d_%H-%M-%S")
folder_path = f"results/{"smooth_dc_penalty"}/{time_str}"
os.makedirs(folder_path, exist_ok=True)

for month in months:
    # function to filter data in the same way for each scenario
    test_df = filter_data(sessions_df, month, year, "all_scheduled")  # .head(30)

    # function to run a scenario (returns a child of BaselineSimulator)
    sim = get_simulator(test_df, "smooth_dc_penalty", verbose=run_verbose)

    # Run simulation and save results and append to the summary
    results_file_name = f"{folder_path}/{month}_{year}_{"smooth_dc_penalty"}.csv"
    summary_file_name = f"{folder_path}/summary.csv"
    aggregate_power_profile_file_name = f"{folder_path}/aggregate_power_profile.csv"
    session_results = generate_session_results(
        sim,
        month,
        results_file_name,
        summary_file_name,
        aggregate_power_profile_file_name,
        visualize=final_verbose,
        verbose=final_verbose,
    )

# Tests peak prediction

In [ ]:
import numpy as np
import json
import pandas as pd
%load_ext autoreload
%autoreload 2
from peak_forecast_simulator import PeakForecastSimulator

In [ ]:
PeakSimulator = PeakForecastSimulator(pd.DataFrame())

In [ ]:
with open("samples_features_peak_pred.json", "r") as f:
    feature_samples = json.load(f)

In [ ]:
def reverse_normalize(PeakSimulator, features):
    reversed_prediction = (
        features
        * (
            PeakSimulator.features_norm_parameters_max
            - PeakSimulator.features_norm_parameters_min
        )
        + PeakSimulator.features_norm_parameters_min
    )
    return reversed_prediction


for sample in feature_samples:
    reversed_sample = reverse_normalize(PeakSimulator, feature_samples[sample])
    feature_samples[sample] = reversed_sample

In [ ]:
sample = feature_samples["sample_13"]
prediction = PeakSimulator.make_prediction(sample, workday=1)
PeakSimulator.visualize_samples(
    time=pd.Timestamp("2023-01-01"), sample=sample, prediction=prediction
)

# Plot results

In [ ]:
import plotly.graph_objects as go

In [ ]:
scenario_to_plot: TypeScenario = "peak_forecast"
run_name = "results_opt_20241219_v2"

In [ ]:
def plot_prices(scenario_to_plot, run_name):
    sim = get_simulator(test_df, scenario_to_plot, verbose=False)
    results = pd.read_csv(
        f"results/{scenario_to_plot}/{run_name}/1_2023_{scenario_to_plot}.csv"
    )
    results["start_time"] = pd.to_datetime(results["start_time"])
    fig = go.Figure()

    fig.add_scatter(
        x=results["start_time"],
        y=results["z_sch"],
        mode="markers",
        name="Scheduled price",
    )
    fig.add_scatter(
        x=results["start_time"],
        y=results["z_reg"],
        mode="markers",
        name="Regular price",
    )

    # add horizontal lines for min and max prices
    fig.add_shape(
        type="line",
        x0=results["start_time"].min(),
        y0=sim.min_price_per_kwh,
        x1=results["start_time"].max(),
        y1=sim.min_price_per_kwh,
        line=dict(color="red", width=2),
    )
    fig.add_shape(
        type="line",
        x0=results["start_time"].min(),
        y0=sim.max_price_per_kwh,
        x1=results["start_time"].max(),
        y1=sim.max_price_per_kwh,
        line=dict(color="red", width=2),
    )

    fig.update_layout(
        title=f"Prices for {scenario_to_plot} scenario (avarege prices (sch, reg) = {results['z_sch'].mean():.2f}, {results['z_reg'].mean():.2f})",
        xaxis_title="Time",
        yaxis_title="Price per kWh",
    )
    fig.show()


plot_prices(scenario_to_plot, run_name)

In [ ]:
plot_prices("all_scheduled", run_name)